# Samsung Cloud Firmware → TWRP Device Tree

This notebook now bundles the **full twrpdtgen-NEXT workflow** into the cloud path.

It can either:

1. download/decrypt Samsung firmware in Colab and return only one extracted `.img`, or
2. continue directly into twrpdtgen, generate the complete candidate TWRP device tree, ZIP it, and download only that ZIP to your computer.

The full Samsung firmware stays in the Colab runtime and is deleted after success unless you explicitly keep it.

In [ ]:
import os, subprocess, sys
repo = '/content/twrpdtgen-NEXT'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/RectifyData/twrpdtgen-NEXT.git', repo], check=True)
else:
    subprocess.run(['git', '-C', repo, 'pull', '--ff-only'], check=True)
os.chdir(repo)
# Bootstrap dependencies in a separate process. Generation below also runs in
# fresh Python processes so rerunning this notebook can never reuse an older
# twrpdtgen module that was already imported by the Colab kernel.
subprocess.run(['bash', 'launcher.sh', '--setup-only'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
version = subprocess.check_output([sys.executable, '-c', 'import twrpdtgen; print(twrpdtgen.__version__)'], text=True).strip()
print(f'twrpdtgen-NEXT {version} @ {commit} ready')


## Firmware and output options

For a normal Samsung device with a dedicated recovery partition, leave `partition = recovery` and `recovery_layout = auto`.
Optional partition-size fields accept verified byte counts or values such as `64MiB`. Leave them blank rather than guessing; the generator intentionally does not derive partition capacity from compressed image length.


In [ ]:
workflow = 'Generate TWRP device tree' #@param ['Generate TWRP device tree', 'Extract image only']
model = 'SM-A207F' #@param {type:'string'}
region = 'NEE' #@param {type:'string'}
partition = 'recovery' #@param {type:'string'}
version = '' #@param {type:'string'}
package = 'AP' #@param ['auto', 'AP', 'BL', 'CP', 'CSC', 'HOME_CSC', 'USERDATA']
threads = 8 #@param {type:'integer'}
skeleton = 'auto' #@param ['auto', 'omni', 'twrp-12.1', 'twrp-14.1']
recovery_layout = 'auto' #@param ['auto', 'dedicated', 'boot', 'vendor_boot']
recovery_root_mode = 'auto' #@param ['auto', 'minimal', 'full']
boot_partition_size = '' #@param {type:'string'}
recovery_partition_size = '' #@param {type:'string'}
super_partition_size = '' #@param {type:'string'}
dynamic_group_size = '' #@param {type:'string'}
download_extracted_image = False #@param {type:'boolean'}
include_stock_root_in_zip = False #@param {type:'boolean'}
keep_full_firmware_in_colab = False #@param {type:'boolean'}


In [ ]:
from pathlib import Path
import subprocess, sys
from google.colab import files

name = Path(partition).name
if name.lower().endswith('.lz4'):
    name = name[:-4]
if not name.lower().endswith(('.img', '.bin')):
    name += '.img'

image_output = Path('/content') / name
tree_output = Path('/content/twrp-tree')
tree_archive = Path('/content/twrp-device-tree.zip')

if workflow == 'Extract image only':
    image_output.unlink(missing_ok=True)
    args = [
        'fetch', '-m', model, '-r', region,
        '--partition', partition,
        '--package', package,
        '-j', str(threads),
        '-o', str(image_output),
        '--overwrite',
    ]
    if version.strip():
        args.extend(['-v', version.strip()])
    if keep_full_firmware_in_colab:
        args.extend(['--keep-firmware', '--work-dir', '/content/twrpdtgen-work'])
    cmd = [sys.executable, '-m', 'twrpdtgen.cloud_slice', *args]
    print('Running fresh image-only process:', ' '.join(cmd))
    rc = subprocess.run(cmd, check=False).returncode
    if rc:
        print('\nCloud slice failed. The detailed cause is printed above.')
    else:
        print('\nImage extraction finished successfully.')
        files.download(str(image_output))
else:
    tree_archive.unlink(missing_ok=True)
    args = [
        '-m', model, '-r', region,
        '--partition', partition,
        '--package', package,
        '-j', str(threads),
        '--tree-output', str(tree_output),
        '--archive', str(tree_archive),
        '--skeleton', skeleton,
        '--recovery-layout', recovery_layout,
        '--recovery-root-mode', recovery_root_mode,
        '--overwrite',
    ]
    if version.strip():
        args.extend(['-v', version.strip()])
    if include_stock_root_in_zip:
        args.append('--include-stock-root')
    for option, value in [
        ('--boot-size', boot_partition_size),
        ('--recovery-size', recovery_partition_size),
        ('--super-size', super_partition_size),
        ('--dynamic-group-size', dynamic_group_size),
    ]:
        if str(value).strip():
            args.extend([option, str(value).strip()])
    if recovery_root_mode == 'full':
        print('WARNING: full recovery-root mode copies stock recovery/init/adbd into the build overlay.')
        print('Use auto/minimal for a normal TWRP skeleton unless the full overlay is intentional.')
    if keep_full_firmware_in_colab:
        args.extend(['--keep-firmware', '--work-dir', '/content/twrpdtgen-work'])
    if download_extracted_image:
        image_output.unlink(missing_ok=True)
        args.extend(['--image-output', str(image_output), '--keep-image'])

    cmd = [sys.executable, '-m', 'twrpdtgen.cloud_tree', *args]
    print('Running fresh full-generation process:', ' '.join(cmd))
    rc = subprocess.run(cmd, check=False).returncode
    if rc:
        print('\nTWRP device-tree generation failed. The detailed cause is printed above.')
    else:
        print('\nGenerated tree remains available under:', tree_output)
        files.download(str(tree_archive))
        if download_extracted_image and image_output.is_file():
            print('Downloading extracted image too:', image_output)
            files.download(str(image_output))


## What the full mode returns

The browser downloads `twrp-device-tree.zip`, containing the generated manufacturer/codename device-tree folder. The expanded tree also remains in `/content/twrp-tree/` for inspection until the Colab runtime ends.

This is still a **candidate source tree**, not proof that a built recovery is safe to flash. Device-specific kernel, encryption, AVB, SELinux, partition, and hardware details still need validation.
By default, the downloadable ZIP excludes `recovery/stock-root/` because it is audit-only, ignored by Git, and can be very large. The expanded stock-root remains in the Colab workspace. Enable `include_stock_root_in_zip` only when you explicitly need that audit payload.

The ZIP preserves Unix symlink metadata. For build use, extract it with a symlink-capable Unix tool so recovery ramdisk links are recreated correctly.

Every extraction/generation run is launched in a fresh Python process from the just-pulled checkout. This prevents Colab's long-lived kernel from mixing old imported twrpdtgen code with newly pulled templates after an update.
